# 05 — Portfolio Analytics

Deep analysis of an optimal portfolio from the DSBPSO optimizer.
Covers reserves, cash flows, NPV distributions, breakeven prices, and sensitivity.

All probabilistic outputs come from price scenario variation — production is deterministic given an allocation.

In [ ]:
import sys
sys.path.insert(0, '..')

import logging
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

from src.orchestrator import run_optimization
from src.optimizer.encoding import decode_particle
from src.data_generation.generator import generate_data
from src.data_generation.profiles import make_oil_profile, logistic_watercut
from src.valuation.capacity import compute_capped_production
from src.valuation.npv import compute_field_npvs
from src.optimizer.evaluation import evaluate_portfolio
from src.price.schwartz_smith import generate_price_scenarios, generate_price_scenarios_final
from src.models import PriceScenarios

logging.basicConfig(level=logging.INFO, format='%(name)s — %(message)s')
print('Imports OK')

## 1. Run optimization & setup

In [ ]:
result = run_optimization(config_path='../config/default.yaml', verbose=True)
archive = result['archive']
config = result['config']
print(f'\nArchive size: {len(archive)}')

In [ ]:
# Regenerate data and decode best solution
field_data, host_data, cost_data = generate_data(config)
n_fields = config['n_fields']
n_hosts = config['n_hosts']
n_years = config['n_years']
max_start_year = config['max_start_year']
max_wells = config['data_generation']['max_wells']
per_well_rate = config['data_generation']['per_well_rate']
val_cfg = config['valuation']

obj_final = np.array([s['objectives_final'] for s in archive])
best_rov_idx = np.argmax(obj_final[:, 0])
alloc = decode_particle(archive[best_rov_idx]['position'], n_fields, n_hosts, max_start_year, max_wells)

print(f'Analyzing: Best ROV solution (E[ROV]={obj_final[best_rov_idx, 0]:.0f}, CVaR={obj_final[best_rov_idx, 1]:.0f} MNOK)')

# Build oil profiles from allocation.n_wells
oil_profiles = np.zeros((n_fields, n_years))
max_rates = np.zeros(n_fields)

for f in range(n_fields):
    if alloc.assignment[f] < 0:
        continue
    n_wells_f = int(alloc.n_wells[f])
    max_rate = n_wells_f * per_well_rate * field_data.complexity_factor[f]
    max_rates[f] = max_rate
    oil_profiles[f] = make_oil_profile(
        n_years, float(field_data.ultimate_recovery[f]),
        max_rate, int(field_data.ramp_years[f]), float(field_data.decline_rate[f]),
    )

# Precompute water cut fractions at field-relative time
t = np.arange(n_years, dtype=float)
wc_fractions = np.zeros((n_fields, n_years))
for f in range(n_fields):
    if alloc.assignment[f] < 0:
        continue
    wc_fractions[f] = logistic_watercut(
        t, float(field_data.wc_midpoint[f]),
        float(field_data.wc_initial[f]),
        float(field_data.wc_final[f]),
        float(field_data.wc_steepness[f]),
    )

# Compute capped production (FCFS oil + derived gas/water)
capped = compute_capped_production(
    oil_profiles, host_data, alloc, n_years,
    max_rates=max_rates,
    ramp_years=field_data.ramp_years,
    gor=field_data.gor,
    wc_fractions=wc_fractions,
)

# Reassemble into (n_fields, n_years) arrays
capped_oil_all = np.zeros((n_fields, n_years))
capped_gas_all = np.zeros((n_fields, n_years))
for host_idx, hr in capped.items():
    for li, fi in enumerate(hr['field_indices']):
        capped_oil_all[fi] = hr['oil'][li]
        capped_gas_all[fi] = hr['gas'][li]

# Generate price scenarios for analysis
prices = generate_price_scenarios_final(config)
print(f'Using {prices.n_scenarios} price scenarios for analysis')

## 2. Reserves per host

Total recoverable oil from capped profiles, grouped by host platform.

In [ ]:
host_colors = ['steelblue', 'tomato', 'seagreen', 'darkorange', 'purple']
years = np.arange(n_years)
developed = [f for f in range(n_fields) if alloc.assignment[f] >= 0]

# Compute reserves per field (sum of capped production over time)
field_oil_reserves = capped_oil_all.sum(axis=1)  # MSm3

fig, ax = plt.subplots(figsize=(10, 5))
active_hosts = sorted(set(int(alloc.assignment[f]) for f in developed))

x = np.arange(len(active_hosts))
bottom = np.zeros(len(active_hosts))

# Stack fields within each host
field_colors = plt.cm.tab10(np.linspace(0, 1, n_fields))
for f in developed:
    h = int(alloc.assignment[f])
    h_pos = active_hosts.index(h)
    reserves_mmbbl = field_oil_reserves[f] * 6.29
    ax.bar(h_pos, reserves_mmbbl, bottom=bottom[h_pos],
           color=field_colors[f], edgecolor='black', label=field_data.names[f])
    if reserves_mmbbl > 1:
        ax.text(h_pos, bottom[h_pos] + reserves_mmbbl / 2,
                f'{reserves_mmbbl:.0f}', ha='center', va='center', fontsize=7)
    bottom[h_pos] += reserves_mmbbl

# Host totals on top
for i, h in enumerate(active_hosts):
    ax.text(i, bottom[i] + 1, f'{bottom[i]:.0f}', ha='center', va='bottom', fontweight='bold', fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels([host_data.names[h] for h in active_hosts])
ax.set_ylabel('Recoverable Oil (mmbbl)')
ax.set_title('Reserves per Host (capped)')
ax.legend(loc='upper right', fontsize=8)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

total_mmbbl = field_oil_reserves.sum() * 6.29
print(f'Total portfolio reserves: {field_oil_reserves.sum():.1f} MSm3 ({total_mmbbl:.0f} mmbbl)')

## 3. CAPEX timeline

When does investment happen? Each field's CAPEX hits at its start year.
In practice you wouldn't develop multiple fields simultaneously — does the optimizer stagger?

In [ ]:
# Compute per-field CAPEX
capex_per_year = np.zeros(n_years)
field_capex = {}

for f in developed:
    h = int(alloc.assignment[f])
    start = int(alloc.start_year[f])
    n_w = int(alloc.n_wells[f])
    capex = float(cost_data.well_cost_per_well[f]) * n_w + cost_data.subsea_base + float(cost_data.pipeline_cost[f, h])
    field_capex[f] = capex
    capex_per_year[start] += capex

fig, ax = plt.subplots(figsize=(12, 5))

# Stacked bar by field
bottom = np.zeros(n_years)
for f in sorted(developed, key=lambda f: alloc.start_year[f]):
    start = int(alloc.start_year[f])
    capex_arr = np.zeros(n_years)
    capex_arr[start] = field_capex[f]
    ax.bar(years, capex_arr, bottom=bottom, color=field_colors[f],
           edgecolor='black', label=f'{field_data.names[f]} ({field_capex[f]:.0f} MNOK)')
    bottom += capex_arr

ax.set_xlabel('Year')
ax.set_ylabel('CAPEX (MNOK)')
ax.set_title('CAPEX Timeline')
ax.legend(loc='upper right', fontsize=8)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

# Summary
for y in range(n_years):
    if capex_per_year[y] > 0:
        fields_this_year = [field_data.names[f] for f in developed if alloc.start_year[f] == y]
        print(f'Year {y}: {capex_per_year[y]:.0f} MNOK ({", ".join(fields_this_year)})')
print(f'Total CAPEX: {sum(field_capex.values()):.0f} MNOK')

## 4. Annual cash flow

Undiscounted portfolio cash flow per year. Revenue varies by price scenario (P10/P50/P90 fan chart).
Costs (OPEX, CAPEX, ABEX) are deterministic.

In [ ]:
barrel = val_cfg['barrel']
btu = val_cfg['btu']
nok_per_usd = val_cfg['nok_per_usd']

# Portfolio-level annual cash flows: (n_scenarios, n_years)
# Revenue: sum across all fields of (oil_prices × capped_oil × barrel × nok_per_usd + gas equivalent)
portfolio_oil = capped_oil_all.sum(axis=0)  # (n_years,)
portfolio_gas = capped_gas_all.sum(axis=0)  # (n_years,)

oil_revenue = prices.oil * (portfolio_oil[np.newaxis, :] * barrel * nok_per_usd)  # (n_scenarios, n_years)
gas_revenue = prices.gas * (portfolio_gas[np.newaxis, :] * btu * nok_per_usd)
revenue = oil_revenue + gas_revenue  # (n_scenarios, n_years)

# Deterministic costs per year
cost_per_year = np.zeros(n_years)
has_production = portfolio_oil > 0

for f in developed:
    h = int(alloc.assignment[f])
    start = int(alloc.start_year[f])

    # Fixed OPEX: shared across fields on this host
    n_on_host = sum(1 for ff in developed if alloc.assignment[ff] == h)
    shared_opex = float(cost_data.opex_rate[h]) / n_on_host

    # Variable OPEX per year for this field
    var_opex = float(cost_data.variable_opex_rate[h]) * capped_oil_all[f] * barrel * nok_per_usd

    # Active years for this field
    field_active = capped_oil_all[f] > 0
    cost_per_year += shared_opex * field_active + var_opex

    # CAPEX at start year
    cost_per_year[start] += field_capex[f]

    # ABEX at last production year
    if field_active.any():
        last_year = int(np.max(np.where(field_active)))
        cost_per_year[last_year] += float(cost_data.abex[h])

# Net cash flow per scenario
net_cashflow = revenue - cost_per_year[np.newaxis, :]  # (n_scenarios, n_years)

# Percentiles
p10 = np.percentile(net_cashflow, 10, axis=0)
p50 = np.percentile(net_cashflow, 50, axis=0)
p90 = np.percentile(net_cashflow, 90, axis=0)

fig, ax = plt.subplots(figsize=(12, 6))
ax.fill_between(years, p10, p90, alpha=0.2, color='steelblue', label='P10–P90')
ax.plot(years, p50, '-', color='steelblue', linewidth=2, label='P50 (median)')
ax.plot(years, np.mean(net_cashflow, axis=0), '--', color='navy', linewidth=1.5, label='Mean')
ax.axhline(0, color='black', linewidth=0.8)
ax.set_xlabel('Year')
ax.set_ylabel('Net Cash Flow (MNOK)')
ax.set_title('Annual Portfolio Cash Flow (undiscounted)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Cumulative cash flow

Running sum of annual cash flows. Shows payback period (when cumulative crosses zero).

In [ ]:
cum_cashflow = np.cumsum(net_cashflow, axis=1)  # (n_scenarios, n_years)

cp10 = np.percentile(cum_cashflow, 10, axis=0)
cp50 = np.percentile(cum_cashflow, 50, axis=0)
cp90 = np.percentile(cum_cashflow, 90, axis=0)

fig, ax = plt.subplots(figsize=(12, 6))
ax.fill_between(years, cp10, cp90, alpha=0.2, color='seagreen', label='P10–P90')
ax.plot(years, cp50, '-', color='seagreen', linewidth=2, label='P50 (median)')
ax.plot(years, np.mean(cum_cashflow, axis=0), '--', color='darkgreen', linewidth=1.5, label='Mean')
ax.axhline(0, color='black', linewidth=0.8)
ax.set_xlabel('Year')
ax.set_ylabel('Cumulative Cash Flow (MNOK)')
ax.set_title('Cumulative Portfolio Cash Flow (undiscounted)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Payback period (median scenario)
payback_idx = np.where(cp50 > 0)[0]
if len(payback_idx) > 0:
    print(f'Median payback period: year {payback_idx[0]}')
else:
    print('Median scenario does not reach payback within planning horizon')

## 6. NPV / ROV distribution

Distribution of portfolio ROV across price scenarios. The CVaR tail (worst 5%) is shaded.

In [ ]:
# Re-evaluate to get per-scenario values with full scenario count
result_rov = evaluate_portfolio(alloc, field_data, host_data, cost_data, prices, config)
scenario_values = result_rov.scenario_values

alpha = val_cfg['cvar_alpha']
var_threshold = np.percentile(scenario_values, alpha * 100)
cvar_val = result_rov.cvar

fig, ax = plt.subplots(figsize=(12, 5))

# Histogram
n_bins = 60
counts, bins, patches = ax.hist(scenario_values, bins=n_bins, color='steelblue',
                                 edgecolor='black', alpha=0.7)

# Shade CVaR tail
for patch, left_edge in zip(patches, bins[:-1]):
    if left_edge < var_threshold:
        patch.set_facecolor('tomato')
        patch.set_alpha(0.8)

ax.axvline(result_rov.expected_rov, color='navy', linestyle='--', linewidth=2,
           label=f'Mean: {result_rov.expected_rov:.0f} MNOK')
ax.axvline(var_threshold, color='red', linestyle=':', linewidth=1.5,
           label=f'VaR {alpha:.0%}: {var_threshold:.0f} MNOK')
ax.axvline(np.percentile(scenario_values, 10), color='gray', linestyle=':', linewidth=1,
           label=f'P10: {np.percentile(scenario_values, 10):.0f} MNOK')
ax.axvline(np.percentile(scenario_values, 90), color='gray', linestyle='--', linewidth=1,
           label=f'P90: {np.percentile(scenario_values, 90):.0f} MNOK')

ax.set_xlabel('Portfolio ROV (MNOK)')
ax.set_ylabel('Count')
ax.set_title('Portfolio ROV Distribution')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'E[ROV]: {result_rov.expected_rov:.0f} MNOK')
print(f'CVaR {alpha:.0%}: {cvar_val:.0f} MNOK')
print(f'Std: {scenario_values.std():.0f} MNOK')
print(f'P(ROV < 0): {(scenario_values < 0).mean():.1%}')

## 7. Exceedance curve

P(portfolio ROV > threshold) — what is the probability of achieving at least a given return?

In [ ]:
sorted_vals = np.sort(scenario_values)
exceedance_prob = 1.0 - np.arange(1, len(sorted_vals) + 1) / len(sorted_vals)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(sorted_vals, exceedance_prob, '-', color='steelblue', linewidth=2)

# Mark key thresholds
ax.axvline(0, color='red', linestyle=':', alpha=0.7, label=f'ROV=0 → P={exceedance_prob[np.searchsorted(sorted_vals, 0)]:.1%}')
ax.axvline(result_rov.expected_rov, color='navy', linestyle='--', alpha=0.7,
           label=f'Mean → P={exceedance_prob[np.searchsorted(sorted_vals, result_rov.expected_rov)]:.1%}')
ax.axhline(0.5, color='gray', linestyle=':', alpha=0.5)

ax.set_xlabel('Threshold (MNOK)')
ax.set_ylabel('P(ROV > threshold)')
ax.set_title('Exceedance Probability Curve')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim(-0.02, 1.02)
plt.tight_layout()
plt.show()

## 8. Breakeven oil price

Scale all oil price paths by a multiplier and compute expected static NPV
(exercise at start year, no LSMC). The breakeven is where expected NPV crosses zero.

In [ ]:
multipliers = np.linspace(0.2, 1.8, 20)
expected_npvs = []

for k in multipliers:
    # Scale oil prices, keep gas unchanged
    scaled_oil = prices.oil * k
    
    # Compute static NPV for each field (exercise at start_year, no LSMC)
    portfolio_npv = np.zeros(prices.n_scenarios)
    
    for host_idx, hr in capped.items():
        n_on_host = len(hr['field_indices'])
        for li, fi in enumerate(hr['field_indices']):
            start = int(alloc.start_year[fi])
            n_w = int(alloc.n_wells[fi])
            capex = field_capex[fi]
            shared_opex = float(cost_data.opex_rate[host_idx]) / n_on_host
            
            npvs = compute_field_npvs(
                capped_oil=hr['oil'][li], capped_gas=hr['gas'][li],
                oil_prices=scaled_oil, gas_prices=prices.gas,
                capex=capex, opex_rate=shared_opex,
                variable_opex_rate=float(cost_data.variable_opex_rate[host_idx]),
                abex=float(cost_data.abex[host_idx]),
                discount_rate=val_cfg['discount_rate'], start_year=start,
                nok_per_usd=nok_per_usd, barrel=barrel, btu=btu,
            )
            # Static NPV: exercise at start_year, discount to year 0
            static_npv = npvs[:, start] * np.exp(-val_cfg['discount_rate'] * start)
            static_npv = np.maximum(static_npv, 0)  # go/no-go optionality
            portfolio_npv += static_npv
    
    expected_npvs.append(np.mean(portfolio_npv))

expected_npvs = np.array(expected_npvs)

# Find breakeven by interpolation
if expected_npvs[0] < 0 and expected_npvs[-1] > 0:
    breakeven_k = np.interp(0, expected_npvs, multipliers)
else:
    breakeven_k = None

# Convert multiplier to approximate USD/bbl (using initial oil price)
base_price = np.exp(config['price']['oil']['E0'])  # ~$71/bbl

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(multipliers * base_price, expected_npvs, 'o-', color='steelblue', linewidth=2, markersize=4)
ax.axhline(0, color='black', linewidth=0.8)
ax.axvline(base_price, color='gray', linestyle='--', alpha=0.5, label=f'Base price ({base_price:.0f} USD/bbl)')

if breakeven_k is not None:
    be_price = breakeven_k * base_price
    ax.axvline(be_price, color='red', linestyle=':', linewidth=1.5,
               label=f'Breakeven: {be_price:.0f} USD/bbl ({breakeven_k:.2f}x)')
    ax.plot(be_price, 0, 'ro', markersize=8, zorder=5)

ax.set_xlabel('Oil Price Level (USD/bbl)')
ax.set_ylabel('Expected Portfolio NPV (MNOK)')
ax.set_title('Breakeven Oil Price (static NPV, no optionality)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

if breakeven_k is not None:
    print(f'Breakeven oil price: {be_price:.0f} USD/bbl ({breakeven_k:.2f}x base)')
else:
    print('No breakeven found in range — portfolio may be profitable across entire range')
    print(f'NPV at lowest price ({multipliers[0]:.1f}x): {expected_npvs[0]:.0f} MNOK')
    print(f'NPV at base price (1.0x): {expected_npvs[np.argmin(np.abs(multipliers - 1.0))]:.0f} MNOK')

## 9. Sensitivity tornado

Perturb one parameter at a time (±20%) and re-evaluate the portfolio.
Shows which inputs have the most impact on expected ROV.

In [ ]:
import copy

# Base case: already computed
base_rov = result_rov.expected_rov

# Define perturbations: (label, low_modifier, high_modifier)
# Each modifier returns (prices_mod, config_mod) for evaluate_portfolio
perturbation = 0.20  # ±20%

sensitivities = {}

# --- Oil price level ---
for direction, factor in [('low', 1 - perturbation), ('high', 1 + perturbation)]:
    scaled_prices = PriceScenarios(oil=prices.oil * factor, gas=prices.gas)
    res = evaluate_portfolio(alloc, field_data, host_data, cost_data, scaled_prices, config)
    sensitivities.setdefault('Oil price', {})[direction] = res.expected_rov

# --- Gas price level ---
for direction, factor in [('low', 1 - perturbation), ('high', 1 + perturbation)]:
    scaled_prices = PriceScenarios(oil=prices.oil, gas=prices.gas * factor)
    res = evaluate_portfolio(alloc, field_data, host_data, cost_data, scaled_prices, config)
    sensitivities.setdefault('Gas price', {})[direction] = res.expected_rov

# --- Discount rate ---
for direction, factor in [('low', 1 - perturbation), ('high', 1 + perturbation)]:
    mod_config = copy.deepcopy(config)
    mod_config['valuation']['discount_rate'] = val_cfg['discount_rate'] * factor
    res = evaluate_portfolio(alloc, field_data, host_data, cost_data, prices, mod_config)
    sensitivities.setdefault('Discount rate', {})[direction] = res.expected_rov

# --- CAPEX level (scale well cost + subsea + pipeline) ---
for direction, factor in [('low', 1 - perturbation), ('high', 1 + perturbation)]:
    from src.models import CostData
    mod_cost = CostData(
        pipeline_cost=cost_data.pipeline_cost * factor,
        subsea_base=cost_data.subsea_base * factor,
        well_cost_per_well=cost_data.well_cost_per_well * factor,
        opex_rate=cost_data.opex_rate,
        variable_opex_rate=cost_data.variable_opex_rate,
        abex=cost_data.abex,
    )
    res = evaluate_portfolio(alloc, field_data, host_data, mod_cost, prices, config)
    sensitivities.setdefault('CAPEX', {})[direction] = res.expected_rov

# --- OPEX level (scale both fixed + variable) ---
for direction, factor in [('low', 1 - perturbation), ('high', 1 + perturbation)]:
    mod_cost = CostData(
        pipeline_cost=cost_data.pipeline_cost,
        subsea_base=cost_data.subsea_base,
        well_cost_per_well=cost_data.well_cost_per_well,
        opex_rate=cost_data.opex_rate * factor,
        variable_opex_rate=cost_data.variable_opex_rate * factor,
        abex=cost_data.abex,
    )
    res = evaluate_portfolio(alloc, field_data, host_data, mod_cost, prices, config)
    sensitivities.setdefault('OPEX', {})[direction] = res.expected_rov

# Build tornado data
labels = []
low_deltas = []
high_deltas = []
for param, vals in sensitivities.items():
    labels.append(param)
    low_deltas.append(vals['low'] - base_rov)
    high_deltas.append(vals['high'] - base_rov)

# Sort by total swing (largest impact on top)
swings = [abs(h - l) for l, h in zip(low_deltas, high_deltas)]
order = np.argsort(swings)[::-1]
labels = [labels[i] for i in order]
low_deltas = [low_deltas[i] for i in order]
high_deltas = [high_deltas[i] for i in order]

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
y_pos = np.arange(len(labels))

for i in range(len(labels)):
    lo, hi = low_deltas[i], high_deltas[i]
    # Low perturbation bar (could be positive or negative)
    ax.barh(y_pos[i], lo, height=0.4, align='center', color='tomato', alpha=0.8,
            label=f'-{perturbation:.0%}' if i == 0 else '')
    # High perturbation bar
    ax.barh(y_pos[i], hi, height=0.4, align='center', color='steelblue', alpha=0.8,
            label=f'+{perturbation:.0%}' if i == 0 else '')

ax.axvline(0, color='black', linewidth=0.8)
ax.set_yticks(y_pos)
ax.set_yticklabels(labels)
ax.set_xlabel('Change in E[ROV] (MNOK)')
ax.set_title(f'Sensitivity Tornado (±{perturbation:.0%} perturbation)')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3, axis='x')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

# Summary table
print(f'{"Parameter":<16} {"−20%":>10} {"Base":>10} {"+20%":>10} {"Swing":>10}')
print('-' * 60)
for i in range(len(labels)):
    lo_val = base_rov + low_deltas[i]
    hi_val = base_rov + high_deltas[i]
    swing = abs(high_deltas[i] - low_deltas[i])
    print(f'{labels[i]:<16} {lo_val:>10.0f} {base_rov:>10.0f} {hi_val:>10.0f} {swing:>10.0f}')